In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans

SEED = 1660

## Читаем данные

In [ ]:
df_train = pd.read_parquet("03_datasets/final_train.parquet")
df_test = pd.read_parquet("03_datasets/final_test.parquet")

In [ ]:
X_train = df_train.drop(columns=["GameId", "Elo"])
Y_train = df_train["Elo"]

X_test = df_test.drop(columns=["GameId", "Elo"])
Y_test = df_test["Elo"]

## Делаем PCA

In [ ]:
mean, std = X_train.mean(), X_train.std()
X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
pca.fit(X_train_scaled)
df_train_clust = pd.DataFrame(pca.transform(X_train_scaled))
df_test_clust = pd.DataFrame(pca.transform(X_test_scaled))
print(pca.explained_variance_ratio_ * 100)

## График

In [ ]:
df_train_clust["Y"] = Y_train.values

In [ ]:
cluster_model = MiniBatchKMeans(n_clusters=150, batch_size=10_000, random_state=SEED, n_init=5)
cluster_model.fit(df_train_clust[[0, 1]])
df_train_clust["cluster"] = cluster_model.predict(df_train_clust[[0, 1]])
df_test_clust["cluster"] = cluster_model.predict(df_test_clust[[0, 1]])

In [ ]:
coloring = (
    df_train_clust
    .groupby("cluster")
    .agg({"Y": "mean"})
    .squeeze().to_dict()
)

df_train_clust["color"] = df_train_clust["cluster"].map(coloring)
df_test_clust["color"] = df_test_clust["cluster"].map(coloring)

In [ ]:
fig = px.scatter(
    df_train_clust.sample(50_000),
    x=0,
    y=1,
    color="color",
#     color="cluster",
#     color="Y",
    color_continuous_scale=["darkred", "red", "orange", "yellow", "lime", "green"]
)

fig.data[0].marker.size=2.5

fig.update_layout(
    template="plotly_dark",
    height=950,
    width=950
)

fig.show()

## Log features

In [ ]:
train_dataset = df_train.drop(columns=["GameId"]).groupby(df_train_clust["cluster"].values).mean()
test_dataset = df_test.drop(columns=["GameId"]).groupby(df_test_clust["cluster"].values).mean()

train_dataset["LogNOkayMoves"] = train_dataset["NOkayMoves"].apply(np.log1p).apply(np.log1p)
train_dataset["LogNEqualGame300"] = train_dataset["NEqualGame300"].apply(np.log1p)
train_dataset["LogMeanCentipawnLoss"] = train_dataset["MeanCentipawnLoss"].apply(np.log1p)
train_dataset["LogStartCentipawnLoss15"] = train_dataset["StartCentipawnLoss15"].apply(np.log1p)
train_dataset["LogKnightCentipawnLoss"] = train_dataset["KnightCentipawnLoss"].apply(np.log1p)
train_dataset["LogPawnCentipawnLoss"] = train_dataset["PawnCentipawnLoss"].apply(np.log1p)
train_dataset["LogMaxAdvLost"] = train_dataset["MaxAdvLost"].apply(np.log1p)
train_dataset["LogStartAdvLost10"] = train_dataset["StartAdvLost10"].apply(np.log1p)

In [ ]:
X_train = train_dataset.drop(columns=["Elo"])
Y_train = train_dataset["Elo"]

X_test = test_dataset.drop(columns=["Elo"])
Y_test = test_dataset["Elo"]

In [ ]:
train_dataset.columns[29]

In [ ]:
px.scatter(
    train_dataset,
    x=train_dataset.columns[30],
    y="Elo",
    trendline="ols"
)

In [ ]:
good_features = [
    "Opening",
    "ECO",
    "NBlunders",
    "LogNOkayMoves",
    "MeanBlunders",
    "MoveNumberBlunder",
    "MoveNumberMistake",
    "MoveNumberBadMove",
    "MeanAbsEval",
    "EvalStd",
    "LogNEqualGame300",
    "LogMeanCentipawnLoss",
    "LogStartCentipawnLoss15",
    "MeanChecks",
    "LogKnightCentipawnLoss",
    "LogPawnCentipawnLoss",
    "WinOddsStd",
    "LogMaxAdvLost",
    "LogStartAdvLost10"
]